# CC-MMD 2026 — Multi-Model Baseline Runner
Zero-shot misogyny classification across multiple models and cultural partitions.

**Supported backends:** HuggingFace · OpenAI · Google Gemini

### Setup
1. **Runtime → Change runtime type → A100 GPU** *(only needed for HuggingFace backend)*
2. **🔑 Secrets → Add your keys:**
   - `HUGGING_FACE_HUB_TOKEN` — for HuggingFace models (Gemma 3 etc.)
   - `OPENAI_API_KEY` — for GPT-4o
3. Upload your dataset or mount Google Drive *(Cell 2)*
4. Set `BACKEND` and `SELECTED_MODELS` in **Cell 4**, then run all cells

In [ ]:

# Install dependencies
!pip install -q git+https://github.com/huggingface/transformers accelerate
!pip install -q 'qwen-vl-utils[decord]==0.0.8'
!pip install -q openai google-generativeai
!pip install -q pandas tqdm Pillow scikit-learn


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 118.0 MB/s eta 0:00:00


In [ ]:
# ── Option A: Mount Google Drive ──────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = "/content/drive/MyDrive/cc_mmd_dataset"

# ── Option B: Upload a zip ────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()
!unzip -q cc_mmd_dataset.zip -d /content/dataset

BASE_DIR   = "/content/dataset"    # root folder containing MDMD/, CMMD/, MAMI/
OUTPUT_DIR = "/content/results"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

Saving cc_mmd_dataset.zip to cc_mmd_dataset.zip


In [ ]:
# ── API Keys from Colab Secrets ───────────────────────────────────────────────
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""


print("HF token  :", "set" if os.environ["HF_TOKEN"] else "NOT SET")


HF token  : set


In [ ]:

# ── CONFIG — edit this cell before running ────────────────────────────────────

# Backend: "huggingface" | "openai"
BACKEND = "huggingface"

# Models to test
SELECTED_MODELS = {
    "huggingface": [
        "Qwen/Qwen2.5-VL-7B-Instruct",   # strong multilingual + vision
        "llava-hf/llava-1.5-7b-hf",       # LLaVA 1.5
    ],
}[BACKEND]

# Partitions to run
RUN_PARTITIONS = [
    "mdmd_original",
    "mdmd_irish",
    "mdmd_chinese",
    "cmmd_original",
    "cmmd_irish",
    "cmmd_indian",
    # "mami_indian",
    # "mami_chinese",
]

SCORE_LOCALLY = True   # compute Macro-F1 + Accuracy if ground truth is available
DELAY         = 1.0    # seconds between API calls (ignored for HuggingFace)

print(f"Backend    : {BACKEND}")
print(f"Models     : {SELECTED_MODELS}")
print(f"Partitions : {RUN_PARTITIONS}")


Backend    : huggingface
Models     : ['Qwen/Qwen2.5-VL-7B-Instruct', 'llava-hf/llava-1.5-7b-hf']
Partitions : ['mdmd_original', 'mdmd_irish', 'mdmd_chinese', 'cmmd_original', 'cmmd_irish', 'cmmd_indian']


In [ ]:
# ── Partition definitions ─────────────────────────────────────────────────────
# Key fix: Western/MAMI uses country="India"/"China" because MAMI labels
# are target_culture1 (Indian) and target_culture2 (Chinese) perspectives.

ALL_PARTITIONS = {
    # MDMD — Tamil/Malayalam Indian memes
    "mdmd_original": {
        "image_dir":   f"{BASE_DIR}/cc_mmd_dataset/MDMD/dev",
        "csv":         f"{BASE_DIR}/cc_mmd_dataset/MDMD/dev.csv",
        "country":     "India",
        "language":    "Tamil",
        "label_col":   "original_labels",
        "description": "MDMD — native Indian perception",
    },
    "mdmd_irish": {
        "image_dir":   f"{BASE_DIR}/cc_mmd_dataset/MDMD/dev",
        "csv":         f"{BASE_DIR}/cc_mmd_dataset/MDMD/dev.csv",
        "country":     "Ireland",
        "language":    "Tamil",
        "label_col":   "irish_labels",
        "description": "MDMD — Irish perception of Indian memes",
    },
    "mdmd_chinese": {
        "image_dir":   f"{BASE_DIR}/cc_mmd_dataset/MDMD/dev",
        "csv":         f"{BASE_DIR}/cc_mmd_dataset/MDMD/dev.csv",
        "country":     "China",
        "language":    "Tamil",
        "label_col":   "chinese_labels",
        "description": "MDMD — Chinese perception of Indian memes",
    },
    # CMMD — Chinese memes
    "cmmd_original": {
        "image_dir":   f"{BASE_DIR}/cc_mmd_dataset/CMMD/dev",
        "csv":         f"{BASE_DIR}/cc_mmd_dataset/CMMD/dev.csv",
        "country":     "China",
        "language":    "Chinese",
        "label_col":   "original_labels",
        "description": "CMMD — native Chinese perception",
    },
    "cmmd_irish": {
        "image_dir":   f"{BASE_DIR}/cc_mmd_dataset/CMMD/dev",
        "csv":         f"{BASE_DIR}/cc_mmd_dataset/CMMD/dev.csv",
        "country":     "Ireland",
        "language":    "Chinese",
        "label_col":   "irish_labels",
        "description": "CMMD — Irish perception of Chinese memes",
    },
    "cmmd_indian": {
        "image_dir":   f"{BASE_DIR}/cc_mmd_dataset/CMMD/dev",
        "csv":         f"{BASE_DIR}/cc_mmd_dataset/CMMD/dev.csv",
        "country":     "India",
        "language":    "Chinese",
        "label_col":   "indian_labels",
        "description": "CMMD — Indian perception of Chinese memes",
    },
    # MAMI — English/Western memes (Indian + Chinese perspectives only)
    "mami_indian": {
        "image_dir":   f"{BASE_DIR}/cc_mmd_dataset/MAMI/dev",
        "csv":         f"{BASE_DIR}/cc_mmd_dataset/MAMI/dev/dev.csv",
        "country":     "India",
        "language":    "English",
        "label_col":   "indian_labels",
        "description": "MAMI — Indian perception of Western memes",
    },
    "mami_chinese": {
        "image_dir":   f"{BASE_DIR}/cc_mmd_dataset/MAMI/dev",
        "csv":         f"{BASE_DIR}/cc_mmd_dataset/MAMI/dev/dev.csv",
        "country":     "China",
        "language":    "English",
        "label_col":   "chinese_labels",
        "description": "MAMI — Chinese perception of Western memes",
    },
}

PARTITIONS = {k: ALL_PARTITIONS[k] for k in RUN_PARTITIONS if k in ALL_PARTITIONS}
print(f"Running {len(PARTITIONS)} partition(s): {list(PARTITIONS.keys())}")

Running 6 partition(s): ['mdmd_original', 'mdmd_irish', 'mdmd_chinese', 'cmmd_original', 'cmmd_irish', 'cmmd_indian']


In [ ]:
# ── Prompt + utilities ────────────────────────────────────────────────────────
import json
import base64
import glob
import time
import csv as csv_module
import pandas as pd
from tqdm import tqdm
from PIL import Image
from io import BytesIO

PROMPT_TEMPLATE = """You will be provided with a {language} meme image.
Your task is to analyze the meme for misogynistic content by examining both
its visual and textual components, interpreted within the cultural, social,
and linguistic context of {country}.

Step-by-Step Instructions:
1. Extract and interpret all visible text from the meme (captions, overlays,
   hashtags, emoji, transliterations, code-mixed text).
2. Analyze the visual elements: characters, expressions, symbols, layout,
   image manipulations.
3. Evaluate against the socio-cultural context of {country} using the
   criteria below.

Classification Criteria:
- Classify as "misogyny" if the content directly or indirectly targets women
  or girls through: stereotyping, hatred, discrimination, sexualization,
  objectification, gender-based violence, insults, victim-blaming, mockery,
  or trivialization — even if conveyed through sarcasm or humor.
- Classify as "not-misogyny" if none of the above criteria apply.

Important:
- Be objective, culturally aware, and precise.
- Do not treat humor or irony as neutral if it carries misogynistic meaning.
{ocr_section}
Respond ONLY in this JSON format (no markdown, no extra text):
{{
  "label": 1,
  "classification": "misogyny",
  "explanation": "Brief explanation referencing image and text with cultural context."
}}
label: 1 = misogyny, 0 = not-misogyny"""

OCR_SECTION = "\nOCR Transcription of meme text:\n\"\"\"\n{transcription}\n\"\"\"\n"


def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def build_prompt(language: str, country: str, transcription: str = None) -> str:
    ocr = OCR_SECTION.format(transcription=transcription) if transcription else ""
    return PROMPT_TEMPLATE.format(language=language, country=country, ocr_section=ocr)


def parse_response(result: str, image_path: str):
    try:
        if "```json" in result:
            result = result.split("```json")[1].split("```")[0].strip()
        elif "```" in result:
            result = result.split("```")[1].strip()
        else:
            start = result.find("{")
            end   = result.rfind("}") + 1
            if start >= 0 and end > start:
                result = result[start:end]

        parsed         = json.loads(result)
        label          = int(parsed.get("label", 0))
        classification = "misogyny" if label == 1 else "not-misogyny"
        explanation    = parsed.get("explanation", "")
        return classification, label, explanation

    except Exception as e:
        print(f"  [WARN] JSON parse failed for {os.path.basename(image_path)}: {e}")
        is_mis = (
            result is not None
            and "misogyny" in result.lower()
            and "not-misogyny" not in result.lower()
            and "not misogyny" not in result.lower()
        )
        label = 1 if is_mis else 0
        return ("misogyny" if is_mis else "not-misogyny"), label, result or "No response."


print("Utilities ready.")

Utilities ready.


In [ ]:

# ── Backend functions ─────────────────────────────────────────────────────────
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

_hf_model     = None
_hf_processor = None
_hf_device    = "cuda" if torch.cuda.is_available() else "cpu"


def load_hf_model(model_name: str):
    global _hf_model, _hf_processor
    if _hf_model is not None:
        del _hf_model
        torch.cuda.empty_cache()

    print(f"Loading {model_name} on {_hf_device}...")
    dtype = torch.bfloat16 if _hf_device == "cuda" else torch.float32

    if "qwen" in model_name.lower():
        from transformers import Qwen2_5_VLForConditionalGeneration
        _hf_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_name, torch_dtype=dtype, device_map="auto",
        ).eval()
        _hf_processor = AutoProcessor.from_pretrained(
            model_name, min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28,
        )

    elif "llava" in model_name.lower():
        from transformers import LlavaForConditionalGeneration
        _hf_model = LlavaForConditionalGeneration.from_pretrained(
            model_name, torch_dtype=dtype, device_map="auto",
        ).eval()
        _hf_processor = AutoProcessor.from_pretrained(model_name)

    else:
        _hf_model = AutoModelForImageTextToText.from_pretrained(
            model_name, torch_dtype=dtype, device_map="auto",
        ).eval()
        _hf_processor = AutoProcessor.from_pretrained(model_name)

    print("Model ready.")


def call_huggingface(model_name: str, prompt: str, base64_image: str) -> str:
    image = Image.open(BytesIO(base64.b64decode(base64_image))).convert("RGB")
    input_len = None

    if "qwen" in model_name.lower():
        from qwen_vl_utils import process_vision_info
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt},
        ]}]
        text = _hf_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = _hf_processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt",
        ).to(_hf_device)
        input_len = inputs["input_ids"].shape[-1]

    elif "llava" in model_name.lower():
        prompt_text = f"USER: <image>\n{prompt}\nASSISTANT:"
        inputs = _hf_processor(
            text=prompt_text, images=image, return_tensors="pt",
        ).to(_hf_device)
        input_len = inputs["input_ids"].shape[-1]

    else:
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt},
        ]}]
        inputs = _hf_processor.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors="pt",
        ).to(_hf_device)
        input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        out_ids = _hf_model.generate(**inputs, max_new_tokens=1024, do_sample=False)

    return _hf_processor.decode(out_ids[0][input_len:], skip_special_tokens=True).strip()


def call_openai(model_name: str, prompt: str, base64_image: str) -> str:
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
        ]}],
        temperature=0.0,
        max_tokens=1024,
    )
    return response.choices[0].message.content


BACKEND_FN = {
    "huggingface": call_huggingface,
    "openai":      call_openai,
}

print("Backend functions ready.")


Backend functions ready.


In [ ]:

# ── classify_image / batch_classify / score_results ───────────────────────────

def norm_id(val):
    """Normalize image_id: strips .0 suffix pandas adds when reading int columns as float."""
    s = str(val).strip()
    return s[:-2] if s.endswith(".0") else s


def classify_image(image_path, model_name, backend, country, language, transcription=None):
    image_id     = os.path.splitext(os.path.basename(image_path))[0]
    prompt       = build_prompt(language, country, transcription)
    base64_image = encode_image(image_path)
    try:
        raw = BACKEND_FN[backend](model_name, prompt, base64_image)
    except Exception as e:
        return {"image_id": image_id, "label": -1,
                "classification": "error", "explanation": str(e), "full_response": str(e)}

    classification, label, explanation = parse_response(raw, image_path)
    return {"image_id": image_id, "label": label,
            "classification": classification, "explanation": explanation, "full_response": raw}


def batch_classify(partition_cfg, model_name, backend, output_dir, delay=0.5):
    image_dir = partition_cfg["image_dir"]
    csv_path  = partition_cfg.get("csv")
    country   = partition_cfg["country"]
    language  = partition_cfg["language"]
    desc      = partition_cfg["description"]

    # Load transcriptions
    transcriptions = {}
    if csv_path and os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        if "transcriptions" in df.columns and "image_id" in df.columns:
            transcriptions = {norm_id(k): str(v)
                              for k, v in zip(df["image_id"], df["transcriptions"])}
            print(f"  Loaded {len(transcriptions)} transcriptions")

    # Collect images
    image_files = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.gif", "*.bmp"]:
        image_files.extend(glob.glob(os.path.join(image_dir, ext)))

    if not image_files:
        print(f"  [WARN] No images in {image_dir} — skipping.")
        return []

    print(f"\n{'='*60}")
    print(f"  Partition : {desc}")
    print(f"  Model     : {model_name} ({backend})")
    print(f"  Country   : {country}  |  Language: {language}")
    print(f"  Images    : {len(image_files)}")
    print(f"{'='*60}")

    safe_model    = model_name.replace("/", "_").replace(":", "-")
    partition_key = partition_cfg["label_col"]
    os.makedirs(output_dir, exist_ok=True)
    out_base   = os.path.join(output_dir, f"{safe_model}__{partition_key}")
    json_out   = f"{out_base}.json"
    txt_out    = f"{out_base}.txt"
    submit_out = f"{out_base}_submission.csv"

    with open(txt_out, "w") as f:
        f.write(f"CC-MMD 2026 Results\nModel: {model_name} | Partition: {desc}\n")
        f.write(f"Country: {country} | Language: {language}\n" + "=" * 60 + "\n\n")

    results = []
    for img_path in tqdm(image_files, desc=f"{model_name[:20]} | {country}", unit="meme", colour="green"):
        image_id      = os.path.splitext(os.path.basename(img_path))[0]
        transcription = transcriptions.get(image_id)

        result = classify_image(img_path, model_name, backend, country, language, transcription)
        results.append(result)

        with open(txt_out, "a") as f:
            f.write(f"Image ID      : {result['image_id']}\n")
            f.write(f"Label         : {result['label']}\n")
            f.write(f"Classification: {result['classification']}\n")
            f.write(f"Explanation   : {result['explanation']}\n")
            f.write("-" * 60 + "\n\n")

        if delay > 0:
            time.sleep(delay)

    with open(json_out, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    with open(submit_out, "w", newline="") as f:
        writer = csv_module.writer(f)
        writer.writerow(["image_id", "label"])
        for r in results:
            writer.writerow([r["image_id"], r["label"]])

    total    = len(results)
    misogyny = sum(1 for r in results if r["label"] == 1)
    not_mis  = sum(1 for r in results if r["label"] == 0)
    errors   = sum(1 for r in results if r["label"] == -1)
    print(f"  Results → {json_out}")
    print(f"  Submit  → {submit_out}")
    print(f"  Summary : total={total}  misogyny={misogyny}  not-misogyny={not_mis}  errors={errors}")
    return results


def score_results(results, csv_path, label_col):
    from sklearn.metrics import f1_score, accuracy_score, classification_report

    df = pd.read_csv(csv_path)
    df["image_id"] = df["image_id"].apply(norm_id)

    def to_int(val):
        if isinstance(val, str):
            return 1 if val.strip().lower() == "misogyny" else 0
        return int(val)

    pred_map = {norm_id(r["image_id"]): r["label"] for r in results if r["label"] != -1}

    y_true, y_pred = [], []
    for _, row in df.iterrows():
        iid = row["image_id"]
        if iid in pred_map and pd.notna(row.get(label_col)):
            y_true.append(to_int(row[label_col]))
            y_pred.append(pred_map[iid])

    if not y_true:
        print("  [WARN] No matching image_ids found for scoring.")
        print(f"  CSV IDs sample   : {df['image_id'].tolist()[:5]}")
        print(f"  Result IDs sample: {list(pred_map.keys())[:5]}")
        return {}

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    accuracy = accuracy_score(y_true, y_pred)

    print(f"  Scoring against '{label_col}'")
    print(f"  Macro-F1 : {macro_f1:.4f}")
    print(f"  Accuracy : {accuracy:.4f}")
    print(f"  Samples  : {len(y_true)}")
    print(classification_report(y_true, y_pred, target_names=["not-misogyny", "misogyny"]))

    return {"macro_f1": macro_f1, "accuracy": accuracy, "n": len(y_true)}


print("All functions ready.")


All functions ready.


In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
all_scores = []
prev_model = None

for model_name in SELECTED_MODELS:
    # Load HuggingFace model once per model (reload if model changes)
    if BACKEND == "huggingface" and model_name != prev_model:
        load_hf_model(model_name)
        prev_model = model_name

    for partition_name, partition_cfg in PARTITIONS.items():
        out_dir = os.path.join(OUTPUT_DIR, partition_name)

        results = batch_classify(
            partition_cfg = partition_cfg,
            model_name    = model_name,
            backend       = BACKEND,
            output_dir    = out_dir,
            delay         = 0 if BACKEND == "huggingface" else DELAY,
        )

        if SCORE_LOCALLY and results:
            csv_path  = partition_cfg.get("csv")
            label_col = partition_cfg["label_col"]
            if csv_path and os.path.exists(csv_path):
                scores = score_results(results, csv_path, label_col)
                if scores:
                    all_scores.append({"model": model_name, "partition": partition_name, **scores})

print("\nAll runs complete.")

Loading Qwen/Qwen2.5-VL-7B-Instruct on cuda...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model ready.
  Loaded 284 transcriptions

  Partition : MDMD — native Indian perception
  Model     : Qwen/Qwen2.5-VL-7B-Instruct (huggingface)
  Country   : India  |  Language: Tamil
  Images    : 284


Qwen/Qwen2.5-VL-7B-I | India:   1%|          | 2/284 [03:23<9:21:48, 119.53s/meme]

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
if all_scores:
    print(f"\n{'='*70}")
    print(f"{'Model':<30} {'Partition':<20} {'Macro-F1':>10} {'Accuracy':>10}")
    print(f"{'-'*70}")
    for s in all_scores:
        print(f"{s['model']:<30} {s['partition']:<20} {s['macro_f1']:>10.4f} {s['accuracy']:>10.4f}")
    print(f"{'='*70}")

    # Save summary
    summary_path = os.path.join(OUTPUT_DIR, "summary.json")
    with open(summary_path, "w") as f:
        json.dump(all_scores, f, indent=2)
    print(f"\nSummary saved → {summary_path}")
else:
    print("No scores computed (either SCORE_LOCALLY=False or no matching ground truth).")

In [ ]:
# ── Download all results as zip ───────────────────────────────────────────────
import shutil
from google.colab import files

zip_path = "/content/cc_mmd_results"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
files.download(f"{zip_path}.zip")
print("Downloaded cc_mmd_results.zip")